In [11]:
import torch
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import cv2
import numpy as np
import os
from sklearn.model_selection import train_test_split

In [12]:
TRAIN_DATASET_PATH = "/Users/sherwinvahidimowlavi/Downloads/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/"
IMG_SIZE = 128
VOLUME_SLICES = 100
VOLUME_START_AT = 22
BATCH_SIZE = 8
NUM_EPOCHS = 35
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

In [ ]:
import os
from sklearn.model_selection import train_test_split

# Fix the BraTS20_Training_355 file naming issue
old_name = os.path.join(TRAIN_DATASET_PATH, "BraTS20_Training_355/W39_1998.09.19_Segm.nii")
new_name = os.path.join(TRAIN_DATASET_PATH, "BraTS20_Training_355/BraTS20_Training_355_seg.nii")

try:
    if os.path.exists(old_name):
        os.rename(old_name, new_name)
        print("✓ File has been renamed successfully!")
    else:
        print("✓ File already renamed or doesn't exist")
except Exception as e:
    print(f"Rename failed: {e}")

# Get all patient directories
train_and_val_directories = [f.path for f in os.scandir(TRAIN_DATASET_PATH) if f.is_dir()]

def pathListIntoIds(dirList):
    x = []
    for i in range(len(dirList)):
        x.append(dirList[i][dirList[i].rfind('/')+1:])
    return x

# Get patient IDs
train_and_test_ids = pathListIntoIds(train_and_val_directories)

# Split data (70% train, 20% val, 10% test)
train_test_ids, val_ids = train_test_split(train_and_test_ids, test_size=0.2, random_state=42)
train_ids, test_ids = train_test_split(train_test_ids, test_size=0.125, random_state=42)  # 0.125 * 0.8 = 0.1

print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

✓ File has been renamed successfully!
Train: 258, Val: 74, Test: 37


In [14]:
import torch
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import cv2
import numpy as np

IMG_SIZE = 128
VOLUME_SLICES = 100
VOLUME_START_AT = 22

class BraTSDataset(Dataset):
    def __init__(self, patient_ids, data_path):
        self.patient_ids = patient_ids
        self.data_path = data_path
        
    def __len__(self):
        return len(self.patient_ids) * VOLUME_SLICES
    
    def __getitem__(self, idx):
        # Determine patient and slice
        patient_idx = idx // VOLUME_SLICES
        slice_idx = idx % VOLUME_SLICES
        
        patient_id = self.patient_ids[patient_idx]
        patient_path = os.path.join(self.data_path, patient_id)
        
        # Load MRI data
        flair = nib.load(os.path.join(patient_path, f'{patient_id}_flair.nii')).get_fdata()
        t1ce = nib.load(os.path.join(patient_path, f'{patient_id}_t1ce.nii')).get_fdata()
        seg = nib.load(os.path.join(patient_path, f'{patient_id}_seg.nii')).get_fdata()
        
        # Extract and resize slice
        slice_num = slice_idx + VOLUME_START_AT
        flair_slice = cv2.resize(flair[:, :, slice_num], (IMG_SIZE, IMG_SIZE))
        t1ce_slice = cv2.resize(t1ce[:, :, slice_num], (IMG_SIZE, IMG_SIZE))
        seg_slice = cv2.resize(seg[:, :, slice_num], (IMG_SIZE, IMG_SIZE), 
                              interpolation=cv2.INTER_NEAREST)
        
        # Stack channels: (2, H, W) format for PyTorch
        image = np.stack([flair_slice, t1ce_slice], axis=0)
        
        # Fix labels: 4 -> 3
        seg_slice[seg_slice == 4] = 3
        
        # Normalize
        image = image / np.max(image) if np.max(image) > 0 else image
        
        # Convert to tensors
        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(seg_slice).long()
        
        return {'image': image, 'mask': mask}

In [15]:
BATCH_SIZE = 8

# Create datasets
train_dataset = BraTSDataset(train_ids, TRAIN_DATASET_PATH)
val_dataset = BraTSDataset(val_ids, TRAIN_DATASET_PATH)
test_dataset = BraTSDataset(test_ids, TRAIN_DATASET_PATH)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")
print(f"✓ Test batches: {len(test_loader)}")

# Test loading one batch
sample_batch = next(iter(train_loader))
print(f"✓ Image batch shape: {sample_batch['image'].shape}")  # Should be (8, 2, 128, 128)
print(f"✓ Mask batch shape: {sample_batch['mask'].shape}")    # Should be (8, 128, 128)

✓ Train batches: 3225
✓ Val batches: 925
✓ Test batches: 463
✓ Image batch shape: torch.Size([8, 2, 128, 128])
✓ Mask batch shape: torch.Size([8, 128, 128])


In [ ]:
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Model
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=2,  # FLAIR + T1CE
    classes=4,       
    activation=None  
).to(DEVICE)

# Loss and optimizer
loss_fn = smp.losses.DiceLoss(mode='multiclass')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
NUM_EPOCHS = 35

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    train_loss = 0.0
    
    for batch_idx, batch in enumerate(train_loader):
        images = batch['image'].to(DEVICE)
        masks = batch['mask'].to(DEVICE)
        
        # Forward
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        # Print progress
        if batch_idx % 50 == 0:
            print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Batch [{batch_idx}/{len(train_loader)}] Loss: {loss.item():.4f}")
    
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(DEVICE)
            masks = batch['mask'].to(DEVICE)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            val_loss += loss.item()
    
    # Epoch summary
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"{'='*60}\n")
    
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
        }, f'checkpoint_epoch_{epoch+1}.pth')
        print(f"✓ Saved checkpoint: checkpoint_epoch_{epoch+1}.pth\n")

Using device: mps
Epoch [1/35] Batch [0/3225] Loss: 0.8967
Epoch [1/35] Batch [50/3225] Loss: 0.5862
Epoch [1/35] Batch [100/3225] Loss: 0.1396
Epoch [1/35] Batch [150/3225] Loss: 0.4150
Epoch [1/35] Batch [200/3225] Loss: 0.5440
Epoch [1/35] Batch [250/3225] Loss: 0.4958
Epoch [1/35] Batch [300/3225] Loss: 0.4860
Epoch [1/35] Batch [350/3225] Loss: 0.3532
Epoch [1/35] Batch [400/3225] Loss: 0.4024
Epoch [1/35] Batch [450/3225] Loss: 0.4627
Epoch [1/35] Batch [500/3225] Loss: 0.4568
Epoch [1/35] Batch [550/3225] Loss: 0.4351
Epoch [1/35] Batch [600/3225] Loss: 0.3784
Epoch [1/35] Batch [650/3225] Loss: 0.1668
Epoch [1/35] Batch [700/3225] Loss: 0.2583
Epoch [1/35] Batch [750/3225] Loss: 0.5647
Epoch [1/35] Batch [800/3225] Loss: 0.4663
Epoch [1/35] Batch [850/3225] Loss: 0.1611
Epoch [1/35] Batch [900/3225] Loss: 0.3733
Epoch [1/35] Batch [950/3225] Loss: 0.1721
Epoch [1/35] Batch [1000/3225] Loss: 0.2442
Epoch [1/35] Batch [1050/3225] Loss: 0.2628
Epoch [1/35] Batch [1100/3225] Loss: 